# Training Results Viewer

This notebook allows you to load and visualize the results of training an autoencoder on the EEG dataset. You can specify which fold's results to load and then view the reconstruction performance of the model on the test set.

### Specify parameters

In [ ]:
# specify which fold to load (0-5)
fold_to_load = 0
model_name = "test"
epochs_to_load = [0]
load_best = True

In [ ]:
import torch
from core.architectures import EEGAutoencoder
from core.datasets import KFoldSplitter
import core.settings as settings
from pathlib import Path
import matplotlib.pyplot as plt

splitter = KFoldSplitter(k_folds=6)
folds = splitter.split_by_subject()


fold = folds[fold_to_load]
tensors = fold.get_datasets()
train_dataset = tensors["train"]
val_dataset = tensors["val"]
test_dataset = tensors["test"]


model = EEGAutoencoder(train_dataset.target_dim).eval()
fold_path = settings.FINETUNE_OUTPUT_DIR / model_name / Path(str(fold_to_load + 1))

checkpoint_paths = [fold_path / f"checkpoint-{epoch}.pth" for epoch in epochs_to_load]

if load_best:
    checkpoint_paths.append(fold_path / "checkpoint-best_test.pth")
    checkpoint_paths.append(fold_path / "checkpoint-best_val.pth")

plt.figure(figsize=(12, 6))


for checkpoint_path in checkpoint_paths:
    if checkpoint_path.exists():
        model.load_state_dict(torch.load(checkpoint_path))
        epoch = test_dataset[0][0]
        test_input = epoch.unsqueeze(0)
        device = next(model.parameters()).device
        test_input = test_input.float().to(device, non_blocking=True) / 100

        with torch.no_grad():
            reconstruction = model(test_input).squeeze(0)

        true_wave = epoch[0, :].cpu().numpy()
        recon_wave = reconstruction[0, :].cpu().numpy()

        # Plot the true and reconstructed waveforms
        i = checkpoint_paths.index(checkpoint_path)
        plt.subplot(len(checkpoint_paths), 1, i + 1)
        plt.plot(true_wave, label='True', alpha=0.7)
        plt.plot(recon_wave, label='Reconstructed', linestyle='--', alpha=0.9)
        plt.title(f"Fold {fold_to_load} - Checkpoint: {checkpoint_path.name}")
        plt.xlabel("Samples")
        plt.ylabel("Normalized Amplitude")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()
